# 04 — Comparação de datasets e recomendação

Este notebook consolida as evidências dos notebooks 01–03 (não redescobre nada, só compara o que já foi medido).


In [1]:
import json

import pandas as pd

with open("../data/experiments/results/eda_laudos_summary.json") as f:
    eda_laudos = json.load(f)
with open("../data/experiments/results/eda_medical_abstracts_summary.json") as f:
    eda_ma = json.load(f)
with open("../data/experiments/results/baseline_results.json") as f:
    baseline = json.load(f)

print("Resumos carregados dos notebooks 01, 02 e 03 (nenhum número novo é inventado aqui).")


Resumos carregados dos notebooks 01, 02 e 03 (nenhum número novo é inventado aqui).


## Tabela comparativa

In [2]:
rows = {
    "Registros (total)": [eda_laudos["n_rows"], eda_ma["n_rows_total"]],
    "Textos únicos": [eda_laudos["n_unique_texts"], eda_ma["n_unique_texts"]],
    "Diversidade textual (únicos/total)": [f"{eda_laudos['text_diversity_ratio']:.1%}",
                                            f"{eda_ma['text_diversity_ratio']:.1%}"],
    "Classes": [", ".join(eda_laudos["classes"]), str(len(eda_ma["classes"])) + " (especialidade clínica)"],
    "Target nativo é urgência?": ["Sim" if eda_laudos["has_native_urgency_target"] else "Não",
                                   "Sim" if eda_ma["has_native_urgency_target"] else "Não"],
    "Desbalanceamento (maior/menor classe)": [f"{eda_laudos['imbalance_ratio']}x", f"{eda_ma['imbalance_ratio']}x"],
    "Tamanho médio do texto (palavras)": [eda_laudos["word_len_mean"], eda_ma["word_len_mean"]],
    "Vazamento treino/teste (split ingênuo/oficial)": [
        f"{baseline['laudos_naive']['n_test']} linhas de teste, "
        f"{eda_laudos['leakage_test_rows_affected_pct']:.1%} com texto duplicado do treino",
        f"{eda_ma['leakage_test_rows_affected_pct']:.1%} das linhas de teste com texto duplicado do treino, "
        f"0% desses pares com rótulo consistente",
    ],
    "Accuracy baseline TF-IDF+RF (split honesto/limpo)": [
        baseline["laudos_dedup"]["accuracy"], baseline["medabs_clean"]["accuracy"],
    ],
    "Macro F1 baseline (split honesto/limpo)": [
        baseline["laudos_dedup"]["macro_f1"], baseline["medabs_clean"]["macro_f1"],
    ],
    "Tempo de treino baseline (s)": [
        baseline["laudos_dedup"]["train_time_s"], baseline["medabs_clean"]["train_time_s"],
    ],
    "Latência de inferência baseline (ms, média)": [
        baseline["laudos_dedup"]["inference_latency_ms_mean"], baseline["medabs_clean"]["inference_latency_ms_mean"],
    ],
    "Facilidade de uso / licenciamento": [
        "Total (gerado internamente, sem restrição)",
        "Alta (CC-BY-SA 3.0, mas requer atribuição; Kaggle indisponível neste ambiente, usado mirror HF)",
    ],
}

comparison = pd.DataFrame(rows, index=["Laudos sintéticos (atual)", "Medical Abstracts TC Corpus"]).T
comparison


,Laudos sintéticos (atual),Medical Abstracts TC Corpus
Registros (total),2000,14438
Textos únicos,107,11227
Diversidade textual (únicos/total),5.3%,77.8%
Classes,"normal, atencao, urgente",5 (especialidade clínica)
Target nativo é urgência?,Sim,Não
Desbalanceamento (maior/menor classe),2.5x,3.22x
Tamanho médio do texto (palavras),5.4,179.9
Vazamento treino/teste (split ingênuo/oficial),"400 linhas de teste, 99.5% com texto duplicado...",35.0% das linhas de teste com texto duplicado ...
Accuracy baseline TF-IDF+RF (split honesto/limpo),1.0,0.7295
Macro F1 baseline (split honesto/limpo),1.0,0.7118


## Qual dataset parece mais adequado para o Tech Challenge, e por quê?

### Dataset recomendado: **Medical Abstracts TC Corpus** como base principal, target = condição médica (5 classes)

**Nota de proveniência:** consultada sobre a falta de target de urgência no Medical Abstracts, a orientação do
professor foi que, para este dataset, **classificar condição/especialidade médica no lugar de urgência é
aceito**. Isso muda o critério de decisão: deixa de ser "qual dataset tem o target certo" (só o sintético tinha)
e passa a ser "qual dataset é a melhor base de treino" nos mesmos critérios de qualquer projeto de ML — volume,
diversidade, realismo. Também cross-checamos os dados baixados diretamente do Kaggle
(`archive(6)/medical_tc_{train,test,labels}.csv`) contra o mirror do Hugging Face usado nos notebooks 01–03: são
**idênticos linha a linha**. Um terceiro arquivo avaliado, `medical_text_classification_fake_dataset.csv`, foi
descartado (só 20 textos únicos repetidos 50× cada, classes que não correspondem ao corpus real).

**1. Por que foi escolhido**

Com a restrição de target removida, a comparação passa a ser decidida por volume, diversidade e realismo — e
nesses três pontos o Medical Abstracts é objetivamente superior ao dataset sintético (ver tabela comparativa
acima).

**2. Evidências que sustentam a escolha**

- **Volume:** 14.438 amostras reais vs. 2.000 linhas sintéticas com só 107 textos únicos.
- **Diversidade textual:** 77,8% de textos únicos (11.227) vs. 5,3% no dataset sintético.
- **Performance honesta (baseline, notebook 03):** 73% accuracy / 0,71 macro F1 no split limpo.
- **Fonte validada:** dados oficiais, licença CC-BY-SA 3.0, confirmados por dois canais independentes (Kaggle
  direto e mirror Hugging Face).

**3. Por que escolher o dataset com números *menores* (73% vs. 100%)**

Olhando só os números, o dataset sintético parece "melhor" — 100% de accuracy contra 73%. Essa leitura é
enganosa. Os 100% do dataset sintético não são evidência de um modelo bom, são evidência de um problema
trivial: só 107 textos únicos (notebook 01), vocabulário quase exclusivo por classe (a palavra "urgente" só
aparece nos templates de `urgente`) — mesmo depois de deduplicar (notebook 03), a accuracy continuou em 100%
porque o classificador só precisa reconhecer 4-6 palavras-gatilho por classe, não entender contexto clínico. É
decoreba de vocabulário, não generalização.

Os 73% do Medical Abstracts vêm da **mesma metodologia** (`Pipeline`, TF-IDF+RandomForest, mesma seed, sem
vazamento — split limpo do notebook 03) aplicada a 14.438 amostras de texto médico real, com vocabulário técnico
genuíno e sobreposição semântica real entre classes correlatas. Como o algoritmo e a metodologia são os mesmos
nos dois casos, a queda de 100% para 73% é inteiramente explicada pela diferença de dado, não de modelo — é a
métrica passando a medir dificuldade real em vez de separabilidade artificial de template. Entre uma métrica
alta mas vazia e uma métrica moderada mas real, a decisão é pela métrica real: 100% num dataset fácil demais não
é uma alegação defensável diante de um avaliador técnico; 73% num corpus real, público e citável é.

**4. Limitações que existem (não escondidas pela escolha)**

- **Tamanho do modelo:** o RandomForest treinado neste corpus tem ~125 MB contra ~0,5 MB no dataset sintético —
  215× maior. Restrição real para a etapa de otimização de latência (Etapa 4 do Tech Challenge); precisa entrar
  na escolha de modelo/técnica de otimização nos notebooks 05–06.
- **Ambiguidade de rótulo no split oficial:** 35% do teste oficial tem texto duplicado do treino com rótulo
  divergente (notebook 02) — a Fase B **precisa** usar a metodologia de split limpo já validada no notebook 03,
  não o split "como veio".
- **Idioma:** o corpus é em inglês; o restante do projeto (README, vídeo, dataset atual) é em português. Não
  invalida o uso experimental, mas é uma decisão que a equipe precisa assumir conscientemente se isso for para
  produção.
- **Desbalanceamento entre classes** (~3,2×) e sobreposição semântica entre especialidades correlatas (ex.:
  `nervous system diseases` teve o pior F1 no baseline) — ponto de partida natural para o notebook 05 (error
  analysis por classe).

**5. O que isso NÃO significa**

Esta é uma recomendação de **dataset e target para os experimentos desta branch**, não uma decisão automática de
migrar a API/Airflow/Docker de produção. Trocar o target de produção de urgência para condição médica é uma
mudança de escopo grande (schemas da API, Airflow, README, roteiro do vídeo) que afeta entregáveis já
construídos — deve ser uma decisão explícita da equipe **depois** da Fase B (comparação de modelos, error
analysis e latência), não algo implementado agora só por causa desta EDA.

**6. Status do dataset sintético**

Não é descartado como conhecimento — os achados dos notebooks 01/03 (vazamento, baixa diversidade) continuam
válidos e relevantes se a equipe decidir manter urgência como target de produção. Mas deixa de ser o dataset
principal desta linha de experimentação.

In [3]:
summary = {
    "recommended_dataset": "medical_abstracts_tc_corpus",
    "recommended_target": "condition_label (5 classes de especialidade médica)",
    "reason": (
        "autorização explícita do professor para aceitar classificação de condição médica no lugar de "
        "urgência neste dataset; sem essa restrição, Medical Abstracts é superior em volume, diversidade "
        "e realismo, e o baseline honesto (split limpo) é uma medida real de dificuldade, não um número "
        "inflado por vazamento/trivialidade como no dataset sintético"
    ),
    "scope_change_authorized_by": "professor (registrado em conversa com a equipe)",
    "production_migration_decided": False,
    "key_evidence": {
        "medabs_volume": eda_ma["n_rows_total"],
        "medabs_text_diversity_ratio": eda_ma["text_diversity_ratio"],
        "medabs_clean_accuracy": baseline["medabs_clean"]["accuracy"],
        "medabs_clean_macro_f1": baseline["medabs_clean"]["macro_f1"],
        "medabs_model_size_kb": baseline["medabs_official"]["model_size_kb"],
        "laudos_naive_accuracy": baseline["laudos_naive"]["accuracy"],
        "laudos_dedup_accuracy": baseline["laudos_dedup"]["accuracy"],
        "laudos_text_diversity_ratio": eda_laudos["text_diversity_ratio"],
        "laudos_model_size_kb": baseline["laudos_naive"]["model_size_kb"],
    },
    "open_limitations": [
        "modelo ~215x maior que o do dataset sintético",
        "35% de overlap ambíguo entre train/test oficiais — usar sempre o split limpo",
        "corpus em inglês, resto do projeto em português",
        "migração de produção (API/Airflow/Docker) ainda não decidida — decisão pós Fase B",
    ],
}

with open("../data/experiments/results/dataset_recommendation.json", "w") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

summary


{'recommended_dataset': 'medical_abstracts_tc_corpus',
 'recommended_target': 'condition_label (5 classes de especialidade médica)',
 'reason': 'autorização explícita do professor para aceitar classificação de condição médica no lugar de urgência neste dataset; sem essa restrição, Medical Abstracts é superior em volume, diversidade e realismo, e o baseline honesto (split limpo) é uma medida real de dificuldade, não um número inflado por vazamento/trivialidade como no dataset sintético',
 'scope_change_authorized_by': 'professor (registrado em conversa com a equipe)',
 'production_migration_decided': False,
 'key_evidence': {'medabs_volume': 14438,
  'medabs_text_diversity_ratio': 0.7776,
  'medabs_clean_accuracy': 0.7295,
  'medabs_clean_macro_f1': 0.7118,
  'medabs_model_size_kb': 128658.9,
  'laudos_naive_accuracy': 1.0,
  'laudos_dedup_accuracy': 1.0,
  'laudos_text_diversity_ratio': 0.0535,
  'laudos_model_size_kb': 598.6},
 'open_limitations': ['modelo ~215x maior que o do dataset